# YOLO11n TFLite Export for ESP32-S3 Deployment

This notebook exports your trained YOLO11n model to **TFLite INT8** format optimized for ESP32-S3 deployment.

**What this notebook does:**
- Exports YOLO11n to TFLite INT8 with proper quantization
- Creates multiple input sizes (128×128, 160×160, 192×192)
- Provides ONNX export for esp-detection workflow
- Includes model validation and deployment guide

**Requirements:**
- Trained YOLO11n model (best.pt)
- Dataset YAML for INT8 calibration
- ESP32-S3-WROOM-1 with 8MB PSRAM

**References:**
- [Ultralytics TFLite Guide](https://docs.ultralytics.com/integrations/tflite/)
- [ESP-Detection Repository](https://github.com/espressif/esp-detection)

## 1. Environment Setup

In [3]:
# Install required packages for TFLite export
import subprocess
import sys

packages = [
    'ultralytics>=8.3.0',  # Latest YOLO11 with TFLite support
    'tensorflow',          # TensorFlow for TFLite conversion
    'onnx>=1.14.0',       # ONNX export
    'onnxruntime',        # ONNX runtime
    'onnxsim',            # ONNX simplification
]

print("📦 Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✅ Installation complete!")

📦 Installing required packages...
✅ Installation complete!


In [ ]:
from ultralytics import YOLO
import torch
import os
from pathlib import Path
import shutil

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Ultralytics: {YOLO.__module__}")

PyTorch version: 2.9.1+cu128
CUDA available: True
Ultralytics: ultralytics.models.yolo.model


## 2. Model Configuration

Set paths to your trained YOLO11n model and configure export parameters.

In [8]:
# Configuration
MODEL_PATH = "runs/detect/yolo11n_leaf_esp32/weights/best.pt"  # Your trained model
OUTPUT_DIR = "esp32_tflite_models"
DATA_YAML = "Dataset.v1.yolov11/data.yaml"  # Your dataset config for calibration

# ESP32-S3 optimized input sizes
ESP32_SIZES = [
    (128, "Maximum speed (~1.5-2.5s)"),
    (160, "Recommended balance (~2-4s)"),
    (192, "Better accuracy (~4-6s)"),
]

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model: {MODEL_PATH}")
print(f"Output: {OUTPUT_DIR}") 
print(f"Dataset: {DATA_YAML}")

Model: runs/detect/yolo11n_leaf_esp32/weights/best.pt
Output: esp32_tflite_models
Dataset: Dataset.v1.yolov11/data.yaml


## 3. Export to ONNX (Recommended for ESP32-S3)

**Important:** Direct TFLite export from YOLO11 has compatibility issues that can crash the kernel. 

**Best practice for ESP32 deployment:**
1. Export to ONNX format (stable and reliable)
2. Use Espressif's esp-detection framework with ONNX models
3. Or convert ONNX → TFLite offline using onnx2tf if needed

In [ ]:
# Verify model exists and load it
print("🔍 Verifying model and environment...")
print(f"Model path: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")

if os.path.exists(MODEL_PATH):
    # Load model for ONNX export (stable workflow)
    model = YOLO(MODEL_PATH)
    print(f"✅ Model loaded successfully")
    print(f"   Model type: YOLO11n")
    print(f"   Classes: {model.names}")
else:
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

In [ ]:
# Suppress TensorFlow warnings that can cause kernel issues
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ONNX EXPORT FOR ESP32-S3 DEPLOYMENT")
print("="*80)

# Check model size
model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"\n📊 Source Model: {MODEL_PATH}")
print(f"   Size: {model_size_mb:.2f} MB (PyTorch)")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()) / 1e6:.2f}M")

onnx_models = {}

# Export to ONNX for all three input sizes
print("\n🎯 Exporting ONNX models for ESP32-S3...")
print("   ONNX is the most reliable format for YOLO11 → ESP32 workflow")

# Export one model at a time to avoid memory issues
for imgsz, description in ESP32_SIZES:
    print(f"\n📦 Exporting ONNX @ {imgsz}×{imgsz}...")
    print(f"   Use case: {description}")
    
    try:
        # Export to ONNX - use minimal options to avoid kernel crash
        # NOTE: simplify=False to avoid onnxslim which triggers TensorFlow
        onnx_path = model.export(
            format="onnx",
            imgsz=imgsz,
            simplify=False,       # Avoid onnxslim - can crash kernel
            opset=12,             # Use opset 12 for better compatibility
            dynamic=False,        # Static shape for embedded deployment
        )
        
        if onnx_path and os.path.exists(str(onnx_path)):
            print(f"   ✅ ONNX export successful!")
            
            # Copy to output directory with descriptive name
            onnx_name = f"yolo11n_{imgsz}.onnx"
            onnx_output = os.path.join(OUTPUT_DIR, onnx_name)
            shutil.copy(str(onnx_path), onnx_output)
            
            onnx_size = os.path.getsize(onnx_output) / (1024 * 1024)
            
            onnx_models[imgsz] = {
                'path': onnx_output,
                'size_mb': onnx_size,
                'description': description,
            }
            
            print(f"   📁 Saved: {onnx_output}")
            print(f"   💾 Size: {onnx_size:.2f} MB")
            
            # Optional: Simplify ONNX manually using onnxsim (safer than built-in)
            try:
                import onnxsim
                import onnx
                print(f"   🔧 Simplifying ONNX model...")
                model_onnx = onnx.load(onnx_output)
                model_simp, check = onnxsim.simplify(model_onnx)
                if check:
                    simplified_path = onnx_output.replace('.onnx', '_simplified.onnx')
                    onnx.save(model_simp, simplified_path)
                    simplified_size = os.path.getsize(simplified_path) / (1024 * 1024)
                    print(f"   ✅ Simplified: {simplified_path} ({simplified_size:.2f} MB)")
                    onnx_models[imgsz]['simplified_path'] = simplified_path
            except Exception as simp_error:
                print(f"   ⚠️  Simplification skipped: {str(simp_error)[:100]}")
        else:
            print(f"   ⚠️  ONNX export failed - file not created")
            
    except Exception as e:
        print(f"   ❌ Export failed: {str(e)[:200]}")
        import traceback
        print(f"   Debug info: {str(e)}")

print("\n" + "="*80)
print("EXPORT SUMMARY")
print("="*80)

if onnx_models:
    print(f"\n✅ Successfully exported {len(onnx_models)} ONNX models")
    print("\n💡 Next Steps for ESP32-S3 Deployment:")
    print("   Option 1 (RECOMMENDED): Use ESP-Detection framework")
    print("      • Clone: https://github.com/espressif/esp-detection")
    print("      • Quantize ONNX → .espdl using esp-ppq")
    print("      • Deploy with ESP-DL on ESP32-S3")
    print()
    print("   Option 2: Convert ONNX → TFLite")
    print("      • Use onnx2tf tool (see next cells)")
    print("      • Apply INT8 quantization")
    print("      • Deploy with TFLite Micro")
else:
    print("\n❌ No models exported successfully")
    print("   Please check:")
    print("   1. Model file exists and is valid")
    print("   2. Ultralytics version >= 8.3.0")
    print("   3. PyTorch is properly installed")

ONNX EXPORT FOR ESP32-S3 DEPLOYMENT

📊 Source Model: runs/detect/yolo11n_leaf_esp32/weights/best.pt
   Size: 5.21 MB (PyTorch)
   Parameters: 2.59M

🎯 Exporting ONNX models for ESP32-S3...
   ONNX is the most reliable format for YOLO11 → ESP32 workflow

📦 Exporting ONNX @ 128×128...
   Use case: Maximum speed (~1.5-2.5s)
Ultralytics 8.3.240 🚀 Python-3.10.12 torch-2.9.1+cu128 CPU (Intel Xeon Gold 5418Y)


YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'runs/detect/yolo11n_leaf_esp32/weights/best.pt' with input shape (1, 3, 128, 128) BCHW and output shape(s) (1, 5, 336) (5.2 MB)

ONNX: starting export with onnx 1.16.0 opset 13...
ONNX: slimming with onnxslim 0.1.80...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/ubuntu/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/ubuntu/.local/lib/python3.10/site-pac

AttributeError: _ARRAY_API not found

SystemError: <built-in function __import__> returned a result with an exception set

: 

## 4. Export Summary & Model Comparison

In [ ]:
import pandas as pd

if onnx_models:
    print("\n✅ Successfully Exported ONNX Models:\n")
    
    summary_data = []
    for imgsz, info in onnx_models.items():
        summary_data.append({
            'Input Size': f"{imgsz}×{imgsz}",
            'ONNX Size': f"{info['size_mb']:.2f} MB",
            'Format': 'ONNX',
            'Use Case': info['description'],
            'Recommendation': '🏆 Best' if imgsz == 160 else ('⚡ Fastest' if imgsz == 128 else '🎯 Accurate'),
        })
    
    df_summary = pd.DataFrame(summary_data)
    print(df_summary.to_string(index=False))
    
    print("\n💡 Deployment Options for ESP32-S3:")
    print("   1. ✅ RECOMMENDED: ESP-Detection workflow (ONNX → ESP-DL)")
    print("      - Clone: https://github.com/espressif/esp-detection")
    print("      - Quantize ONNX to .espdl using esp-ppq")
    print("      - Deploy with ESP-DL framework (optimized for ESP32)")
    print()
    print("   2. Alternative: Convert ONNX to TFLite")
    print("      - Use onnx2tf tool (installed)")
    print("      - Apply INT8 quantization")
    print("      - Deploy with TFLite Micro")
    
    print("\n🎯 Recommended Model for ESP32-S3:")
    if 160 in onnx_models:
        print(f"   📦 ONNX: {onnx_models[160]['path']}")
        print(f"   💾 Size: {onnx_models[160]['size_mb']:.2f} MB (FP32)")
        print(f"   ⏱️  Expected: ~2-4 seconds per inference (after INT8 quantization)")
        print(f"   🧠 Memory: ~2-3 MB (quantized model + activations)")
    
else:
    print("\n❌ No models exported successfully")
    print("   Check model path and configuration in previous cells")

print("\n" + "="*80)


❌ No TFLite models exported successfully
   Check model path and dataset configuration



## 4b. Convert ONNX to TFLite (Optional)

If you want to use TFLite instead of ESP-DL, you can convert the ONNX models to TFLite format.

In [ ]:
# Convert ONNX to TFLite using onnx2tf
print("="*80)
print("ONNX → TFLITE CONVERSION (INT8)")
print("="*80)

if not tflite_models:
    print("\n⚠️  No ONNX models to convert. Run the export cell first.")
else:
    # Convert the 160x160 model (recommended)
    if 160 in tflite_models:
        onnx_file = tflite_models[160]['onnx_path']
        output_name = "yolo11n_int8_160"
        
        print(f"\n📦 Converting {onnx_file} to TFLite...")
        print("   This may take a few minutes...")
        
        try:
            import subprocess
            
            # Use onnx2tf command line tool
            cmd = [
                "onnx2tf",
                "-i", onnx_file,
                "-o", os.path.join(OUTPUT_DIR, "tflite_converted"),
                "-oiqt",  # Output INT8 quantized TFLite
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(f"   ✅ Conversion successful!")
                
                # Find the generated TFLite file
                tflite_dir = os.path.join(OUTPUT_DIR, "tflite_converted")
                if os.path.exists(tflite_dir):
                    tflite_files = [f for f in os.listdir(tflite_dir) if f.endswith('.tflite')]
                    if tflite_files:
                        for tflite_file in tflite_files:
                            tflite_path = os.path.join(tflite_dir, tflite_file)
                            size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
                            print(f"   📁 {tflite_file}")
                            print(f"   💾 Size: {size_mb:.2f} MB")
            else:
                print(f"   ⚠️  Conversion failed:")
                print(f"   {result.stderr[:500]}")
                print("\n💡 Alternative: Use the ONNX models with esp-detection workflow")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
            print("\n💡 For ESP32-S3, we recommend using the ONNX model with esp-detection")
    
    print("\n" + "="*80)

## 5. Alternative: Export via ESP-Detection ONNX Workflow

For integration with Espressif's esp-detection and ESP-DL framework.

In [ ]:
print("="*80)
print("ONNX INT8 EXPORT (ESP-DETECTION WORKFLOW)")
print("="*80)

# Load model
model = YOLO(MODEL_PATH)

onnx_models = {}

for imgsz, description in ESP32_SIZES:
    print(f"\n📦 Exporting ONNX INT8 @ {imgsz}×{imgsz}...")
    print(f"   Use case: {description}")
    
    try:
        # Export to ONNX with INT8
        export_path = model.export(
            format="onnx",
            imgsz=imgsz,
            int8=True,
            data=DATA_YAML,
            simplify=True,
            opset=13,  # ESP-PPQ uses opset 13
        )
        
        if os.path.exists(export_path):
            onnx_size = os.path.getsize(export_path) / (1024 * 1024)
            
            new_name = f"yolo11n_int8_{imgsz}.onnx"
            new_path = os.path.join(OUTPUT_DIR, new_name)
            shutil.copy(export_path, new_path)
            
            onnx_models[imgsz] = {
                'path': new_path,
                'size_mb': onnx_size
            }
            
            print(f"   ✅ SUCCESS!")
            print(f"   📁 File: {new_path}")
            print(f"   💾 Size: {onnx_size:.2f} MB")
        else:
            print(f"   ⚠️  Export file not found")
            
    except Exception as e:
        print(f"   ❌ Export failed: {str(e)[:150]}")

if onnx_models:
    print("\n✅ ONNX models exported successfully!")
    print("\n📖 Next Steps for ESP-DL Deployment:")
    print("   1. Use esp-ppq to convert ONNX → ESP-DL format (.espdl)")
    print("   2. Follow esp-detection quantization workflow")
    print("   3. Deploy using ESP-IDF with ESP-DL library")
    print("\n🔗 Reference: https://github.com/espressif/esp-detection")

print("\n" + "="*80)

## 6. Model Validation (Optional)

Test the exported TFLite model before deployment.

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
from PIL import Image

def test_tflite_model(tflite_path, test_image_path, imgsz=160):
    """
    Test a TFLite model with a sample image.
    """
    print(f"\n🧪 Testing TFLite model: {tflite_path}")
    
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    # Get input/output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"   Input shape: {input_details[0]['shape']}")
    print(f"   Input dtype: {input_details[0]['dtype']}")
    print(f"   Output tensors: {len(output_details)}")
    
    # Load and preprocess test image
    img = cv2.imread(test_image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (imgsz, imgsz))
    
    # Prepare input (INT8: 0-255)
    input_data = np.expand_dims(img_resized, axis=0).astype(np.uint8)
    
    # Run inference
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    
    # Get output
    output = interpreter.get_tensor(output_details[0]['index'])
    
    print(f"   Output shape: {output.shape}")
    print(f"   ✅ Inference successful!")
    
    return output

# Test with a sample image
if tflite_models and 160 in tflite_models:
    test_img = "Dataset.v1.yolov11/test/images/IMG_2906_JPG.rf.7b0c4c80b4b6f62c4e0f5e4b3cca1d95.jpg"  # Use your test image
    if os.path.exists(test_img):
        result = test_tflite_model(tflite_models[160]['path'], test_img, imgsz=160)
    else:
        print(f"⚠️  Test image not found: {test_img}")

## 7. ESP32-S3 Deployment Guide

### Hardware Requirements
- **ESP32-S3-WROOM-1-N8R8** (8MB PSRAM) ← **Mandatory**
- 16MB Flash memory
- Camera module (OV2640, OV5640, or ESP32-S3-EYE)

### Deployment Steps

#### Option 1: TensorFlow Lite Micro
```bash
# 1. Convert TFLite to C array
xxd -i yolo11n_int8_160.tflite > model_data.h

# 2. Use TFLite Micro runtime
# Include in your ESP-IDF project
```

#### Option 2: ESP-TFLite-Micro (Recommended)
```bash
# Clone ESP-TFLite-Micro
git clone https://github.com/espressif/esp-tflite-micro

# Follow examples in esp-tflite-micro/examples/
```

#### Option 3: ESP-Detection + ESP-DL
```bash
# Use the ONNX models with esp-detection workflow
# Quantize to .espdl format using esp-ppq
# Deploy with ESP-DL library
```

### Performance Expectations
- **128×128**: ~1.5-2.5 seconds per inference
- **160×160**: ~2-4 seconds per inference (recommended)
- **192×192**: ~4-6 seconds per inference

### Memory Usage
- Model: ~1.9 MB (INT8)
- Runtime: ~2-4 MB (activations + buffers)
- **Total: 8MB PSRAM required**

## 8. Final Summary

In [ ]:
print("="*80)
print("📦 EXPORT COMPLETE - MODELS READY FOR ESP32-S3 DEPLOYMENT")
print("="*80)

print("\n📁 Exported Models Location:")
print(f"   Directory: {os.path.abspath(OUTPUT_DIR)}")

if onnx_models:
    print("\n✅ ONNX Models (Ready for ESP-Detection):")
    for imgsz, info in onnx_models.items():
        print(f"   • {imgsz}×{imgsz}: {info['path']} ({info['size_mb']:.2f} MB)")

print("\n🎯 Recommended for ESP32-S3:")
if 160 in onnx_models:
    print(f"   📦 Model: {onnx_models[160]['path']}")
    print(f"   💾 Size: {onnx_models[160]['size_mb']:.2f} MB (FP32 ONNX)")
    print(f"   ⏱️  Inference: ~2-4 seconds @ 160×160 (after quantization)")
    print(f"   🧠 Memory: ~2-3 MB (INT8 quantized)")

print("\n📖 Deployment Workflow:")
print("   Step 1: Clone ESP-Detection")
print("      git clone https://github.com/espressif/esp-detection")
print()
print("   Step 2: Quantize ONNX to ESP-DL format")
print("      cd esp-detection/tools/quantization")
print("      python quantize_onnx.py --model yolo11n_160.onnx")
print()
print("   Step 3: Deploy to ESP32-S3")
print("      - Copy .espdl model to ESP-IDF project")
print("      - Use ESP-DL inference API")
print("      - Flash to ESP32-S3-WROOM-1 (8MB PSRAM required)")

print("\n📚 Additional Resources:")
print("   • ESP-Detection: https://github.com/espressif/esp-detection")
print("   • ESP-DL Docs: https://docs.espressif.com/projects/esp-dl/")
print("   • YOLO11 Export: https://docs.ultralytics.com/modes/export/")

print("\n" + "="*80)
print("🎉 Export complete! Your YOLO11n model is ready for ESP32-S3!")
print("="*80)